# 🫀 Chest Pain Lab — From Patient Data to a (Careful) AI Model
**WFT Digitale Medizin · Hands-on lab (~60 minutes)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoLue/wft_digital_medicine/blob/main/chest_pain_lab.ipynb)

**The scenario.** It's a busy night in the emergency department. Patients arrive with chest pain and related symptoms. We have data from **918 patients** — age, blood pressure, ECG findings, exercise test results — and whether they turned out to have **heart disease**.

**Your job today:** build a small prediction model *and* find out **when you should not trust it**.

> ⚠️ For teaching only. Historical, partly 30+ year old data. Never use this for real patient decisions.

## How this lab works

| | |
|---|---|
| 🟢 **Core** | Everyone. Just run the cells and answer the questions. No coding experience needed. |
| 🟡 **Explore** | Change a number or a word and see what happens. |
| 🔴 **Challenge** | Write a few lines of code yourself. Solutions are hidden under ▶ *Solution*. |
| 🔮 **Predict first!** | Before running a 🔮 cell, write down your guess. Wrong guesses are where the learning happens. |

**Running code:** click into a cell and press **Shift + Enter**. Run cells **top to bottom**.
**Errors are normal.** Read the *last line* of the red message — it usually tells you what's wrong.

| Exercise | Topic | ⏱ |
|---|---|---|
| 0 | Open the notebook + warm-up: Python in 5 minutes | 7 min |
| 1 | Meet the data (and its problems) | 10 min |
| ⏸ | Checkpoint 1 — eyes to the front | 5 min |
| 2 | Train your first model | 10 min |
| 3 | Is the model *good*? Thresholds & prevalence | 10 min |
| ⏸ | Checkpoint 2 — eyes to the front | 5 min |
| 4 | Why does it decide that? Explanations & fairness | 9 min |
| ⭐ | Bonus: Mini-RAG · AI as coding assistant | open end |

**Done early?** Do the 🟡/🔴 tasks

---
# Exercise 0 · Warm-up: Python in 5 minutes 🟢
First, run the setup cell. It loads the libraries (toolboxes) we need.

In [ ]:
# Setup: load the toolboxes (run me first!)
import pandas as pd                      # tables ("Excel for Python")
import numpy as np                       # numbers & maths
import matplotlib.pyplot as plt          # plots
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

plt.rcParams["figure.figsize"] = (7, 4)
print("✅ Setup done")

A **variable** stores a value. A **dictionary** stores a small record — like one row of a patient chart.

In [ ]:
patient = {"age": 58, "sex": "M", "resting_bp": 150, "cholesterol": 270}

print("Age:", patient["age"])
print("Blood pressure:", patient["resting_bp"], "mmHg")

A **function** is a reusable recipe. This one encodes a *hand-written rule* — exactly how the first "expert systems" in medicine worked in the 1970s/80s.

In [ ]:
def classify_bp(systolic):
    if systolic >= 140:
        return "hypertensive range"
    elif systolic >= 120:
        return "elevated"
    else:
        return "normal"

print(classify_bp(patient["resting_bp"]))
print(classify_bp(118))

🟡 **Explore:** change `118` to other values and re-run.

🔴 **Challenge:** add a category `"hypertensive crisis range"` for values ≥ 180. (Hint: the order of the `if` checks matters!)

<details><summary>Solution</summary>

```python
def classify_bp(systolic):
    if systolic >= 180:
        return "hypertensive crisis range"
    elif systolic >= 140:
        return "hypertensive range"
    elif systolic >= 120:
        return "elevated"
    else:
        return "normal"
```
</details>

💡 **Key idea:** Here *we* wrote the rule. In **machine learning**, the computer *learns* the rules from examples. That's what we do next.

---
# Exercise 1 · Meet the data (and its problems)

In [ ]:
URL = "https://raw.githubusercontent.com/MoLue/wft_digital_medicine/main/data/heart.csv"
df = pd.read_csv(URL)

print(df.shape)   # (rows, columns)
df.head()         # first 5 patients

| Column | Meaning |
|---|---|
| `Age` | years |
| `Sex` | M / F |
| `ChestPainType` | TA = typical angina · ATA = atypical angina · NAP = non-anginal pain · ASY = asymptomatic |
| `RestingBP` | resting systolic blood pressure (mmHg) |
| `Cholesterol` | serum cholesterol (mg/dl) |
| `FastingBS` | fasting blood sugar > 120 mg/dl (1 = yes) |
| `RestingECG` | Normal · ST (ST-T abnormality) · LVH (left ventricular hypertrophy) |
| `MaxHR` | maximum heart rate during exercise test |
| `ExerciseAngina` | angina during exercise (Y/N) |
| `Oldpeak` | ST depression during exercise (mm) |
| `ST_Slope` | slope of the peak exercise ST segment: Up · Flat · Down |
| **`HeartDisease`** | **the label we want to predict: 1 = heart disease, 0 = normal** |

### 🔮 Predict
What percentage of these patients have heart disease? Write your guess *before* running the next cell.

✍️ **Your answer – my guess in %:** *(double-click to write)*

…

In [ ]:
df["HeartDisease"].value_counts(normalize=True).round(3)

🟢 **Question:** Is this a typical share for *all* chest-pain patients in an emergency department? What does that tell you about **how these patients were selected**? (Keep this in mind for Exercise 3!)

✍️ **Your answer:** *(double-click to write)*

…

In [ ]:
# Age distribution: patients with vs. without heart disease
df[df.HeartDisease == 0]["Age"].plot.hist(bins=20, alpha=0.6, label="no heart disease")
df[df.HeartDisease == 1]["Age"].plot.hist(bins=20, alpha=0.6, label="heart disease")
plt.xlabel("Age (years)"); plt.legend(); plt.show()

### 🟡 Explore: a data quality detective task
Replace `"Age"` in the cell above with `"Cholesterol"` and re-run. Look closely at the left edge of the plot. What do you see? Compare the two distributions: does every plotted value look like a plausible measurement? The summary statistics below give another view.

In [ ]:
df.describe().round(1)   # summary statistics for every numeric column

🟢 **Question:** Look at the `min` row. Which values are *physiologically impossible*? What do you think they really mean?

✍️ **Your answer:** *(double-click to write)*

…

In [ ]:
print("Patients with Cholesterol == 0:", (df.Cholesterol == 0).sum())

<details><summary>What's going on?</summary>

A cholesterol of 0 mg/dl is impossible. These are **missing values** that were stored as `0`. A model doesn't know that — it will happily learn "cholesterol = 0 ⇒ something". Data cleaning is often 80% of the real work.
</details>

### 🔴 Challenge: who is in the data?
Find out (1) how many women and men are in the dataset, and (2) the heart-disease rate for each group.
Hint: `df["Sex"].value_counts()` and `df.groupby("Sex")["HeartDisease"].mean()`

In [ ]:
# your code here

<details><summary>Solution</summary>

```python
print(df["Sex"].value_counts())
print(df.groupby("Sex")["HeartDisease"].mean().round(2))
```
Only about **one in five patients is a woman**. Keep that in mind for Exercise 4.
</details>

---
# Exercise 2 · Train your first model

Computers only understand numbers. Text columns like `ChestPainType = "ASY"` are turned into 0/1 columns (*one-hot encoding*). Each new column answers a yes/no question, e.g. "Is this patient's chest pain type ASY?" We keep the outcome (`HeartDisease`) separate: it is what the model must predict, not an input it may inspect.

In [ ]:
X = pd.get_dummies(df.drop(columns="HeartDisease"), drop_first=True).astype(float)   # features
y = df["HeartDisease"]                                                              # label

print(X.shape)
X.head(3)

### Training data vs. test data
Like an exam: you learn from the **training set** (75 %), and you're graded on **new questions** you have never seen -> the **test set** (25 %). `stratify=y` keeps roughly the same disease proportion in both sets; `random_state=42` makes the split reproducible. A test score is still an estimate on this historical dataset, not proof the model will work in today's ED.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print("Training patients:", len(X_train))
print("Test patients:    ", len(X_test))

### 🔮 Predict
What **accuracy** (share of correct predictions on the test set) do you expect? 50 %? 70 %? 90 %?

✍️ **Your answer – my guess:** *(double-click to write)*

…

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)                    # <- this is the "learning"

accuracy = model.score(X_test, y_test)
print(f"Accuracy on unseen test patients: {accuracy:.1%}")

That's it — **three lines** of code trained the model. The hard part is everything around it.

### 🟡 Explore: a model you can read
A **decision tree** learns `if … then …` rules — like the rule we wrote by hand in Exercise 0, but learned from data.
Try `max_depth = 1`, `2`, `3`, `6`, `None`. What happens to train vs. test accuracy?

In [ ]:
max_depth = 3   # <- change me

tree = DecisionTreeClassifier(max_depth=max_depth, random_state=42).fit(X_train, y_train)
print(f"Train accuracy: {tree.score(X_train, y_train):.1%}")
print(f"Test accuracy:  {tree.score(X_test, y_test):.1%}")

if max_depth is not None and max_depth <= 3:
    plt.figure(figsize=(18, 7))
    plot_tree(tree, feature_names=X.columns, class_names=["normal", "disease"],
              filled=True, fontsize=9, impurity=False)
    plt.show()

🟢 **Question:** With `max_depth=None` the tree is nearly perfect on the *training* data. Why isn't that good news? (Keyword: **overfitting** — learning by heart instead of understanding.)

✍️ **Your answer:** *(double-click to write)*

…

### 🔴 Challenge: which information matters?
Use the helper below to retrain **without** certain columns. Which column, when removed, hurts the most? Does removing `Sex_M` change much?

In [ ]:
def train_without(columns_to_drop):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(X_train.drop(columns=columns_to_drop), y_train)
    return m.score(X_test.drop(columns=columns_to_drop), y_test)

print(list(X.columns))
print(f"Without ST_Slope: {train_without(['ST_Slope_Flat', 'ST_Slope_Up']):.1%}")
# try your own: train_without(['Age']), train_without(['Sex_M']), ...

---
# Exercise 3 · Is the model *good*?

Accuracy alone hides the most important question in medicine: **what kind of mistakes does the model make?**

The model doesn't say "yes/no" — it gives a **predicted probability**. *We* choose the **threshold** above which we call a patient "positive". At 0.5, a predicted 0.6 counts as positive; at 0.7, the same patient counts as negative. These numbers are model estimates, not a clinical diagnosis.

In [ ]:
proba = model.predict_proba(X_test)[:, 1]   # probability of heart disease for each test patient

print("First 5 test patients – predicted risk:", np.round(proba[:5], 2))
print("What actually happened:               ", y_test.values[:5])

In [ ]:
def evaluate(threshold=0.5):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    ppv  = tp / (tp + fp) if (tp + fp) else float("nan")
    print(f"Threshold {threshold:.2f}")
    print(f"                  actually ill   actually healthy")
    print(f"  predicted ill        {tp:4d} (TP)      {fp:4d} (FP)")
    print(f"  predicted healthy    {fn:4d} (FN)      {tn:4d} (TN)")
    print()
    print(f"  Sensitivity (recall): {sens:.1%}   ← share of ill patients we catch")
    print(f"  Specificity:          {spec:.1%}   ← share of healthy patients we correctly reassure")
    print(f"  PPV (precision):      {ppv:.1%}   ← if the model says 'ill', how often is that true?")
    print(f"  Missed patients (FN): {fn}")

evaluate(0.5)

### 🟢 Use the slider
Move the threshold. (If the slider doesn't appear, just call `evaluate(0.3)` etc. in a new cell.) A **false negative (FN)** is an ill patient missed by the model; a **false positive (FP)** is a healthy patient flagged as ill. Lowering the threshold usually catches more ill patients but also raises more false alarms.

**Tasks**
1. **Screening / rule-out** in the ED: find the threshold where sensitivity is **≥ 95 %**. What does that cost in specificity?
2. **Confirmation before an invasive procedure:** find a threshold where PPV is **≥ 95 %**. How many ill patients do we now miss?

In [ ]:
from ipywidgets import interact, FloatSlider
interact(evaluate, threshold=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5));

✍️ **Your answer – thresholds we chose and why:** *(double-click to write)*

…

### 🟡 Explore: the prevalence trap
**Prevalence** means the share of people who actually have the disease. In this selected dataset ~55 % are labelled ill; that is not the rate in everyone with chest pain or in the general population. Try hypothetical prevalence values of 5 % and 1 % below.

**The same model** (same sensitivity and specificity) — what happens to the PPV? 🔮 Predict first, then move the slider.

In [ ]:
def ppv_at_prevalence(prevalence=0.55, sensitivity=0.92, specificity=0.84):
    ppv = sensitivity * prevalence / (sensitivity * prevalence + (1 - specificity) * (1 - prevalence))
    per_1000 = 1000 * prevalence
    true_pos = sensitivity * per_1000
    false_pos = (1 - specificity) * (1000 - per_1000)
    print(f"Prevalence {prevalence:.1%}  →  PPV = {ppv:.1%}")
    print(f"Out of 1000 people: {true_pos:.0f} true alarms vs. {false_pos:.0f} false alarms")

interact(ppv_at_prevalence,
         prevalence=FloatSlider(min=0.01, max=0.60, step=0.01, value=0.55),
         sensitivity=FloatSlider(min=0.5, max=1.0, step=0.01, value=0.92),
         specificity=FloatSlider(min=0.5, max=1.0, step=0.01, value=0.84));

🟢 **Question:** A start-up advertises *"92 % sensitivity, 84 % specificity!"* for a screening app used by the general public. What do you ask them?

✍️ **Your answer:** *(double-click to write)*

…

### 🔴 Challenge: ROC curve
**ROC** stands for *receiver operating characteristic*. It plots **sensitivity** (true positive rate) against **1 - specificity** (false positive rate) as the threshold changes. Each point is a different trade-off between catching disease and raising false alarms.

**AUC** is the *area under the ROC curve*: a summary of how well the model ranks ill patients above healthy ones across thresholds. 0.5 is roughly chance; 1.0 is perfect ranking. Neither ROC nor AUC tells us which threshold is safe for a clinical decision, or whether the probabilities themselves are reliable.

Plot the ROC curve and compute the AUC (Hint: `roc_curve(y_test, proba)` and `roc_auc_score(y_test, proba)`).

In [ ]:
# your code here

<details><summary>Solution</summary>

```python
fpr, tpr, thresholds = roc_curve(y_test, proba)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, proba):.2f}")
plt.plot([0, 1], [0, 1], "--", color="grey", label="coin flip")
plt.xlabel("1 – Specificity (false positive rate)"); plt.ylabel("Sensitivity")
plt.legend(); plt.show()
```
</details>

---
# Exercise 4 · Why does it decide that?
In medicine, a prediction is not enough — we need to know **why**.
Logistic regression gives every feature a **weight**: positive = pushes towards "disease", negative = pushes towards "normal". The pipeline standardizes numeric inputs before fitting, so these weights describe changes on that standardized scale. A weight is an association learned from this dataset, not proof that a feature causes disease.

In [ ]:
weights = pd.Series(model[-1].coef_[0], index=X.columns).sort_values()
weights.plot.barh(color=np.where(weights > 0, "tab:red", "tab:blue"), figsize=(7, 5))
plt.title("What pushes the prediction? (red = towards disease)"); plt.show()

🟢 **Question:** Which features push most strongly towards *disease*? Does that match clinical knowledge? Anything surprising? *(Look at `Cholesterol` — and remember the detective task in Exercise 1.)*

✍️ **Your answer:** *(double-click to write)*

…

### 🟢 Explain one patient
For one patient we can split the model's *score* into **contributions of each feature**. Positive bars push towards disease; negative bars push away. They add up on the model's log-odds scale, **not** as percentages of risk. This resembles the idea behind explanation methods like **SHAP** (from the lecture), but is not a SHAP calculation or a causal explanation.

In [ ]:
patient_nr = 0   # 🟡 try other numbers (0 … 229)

x_scaled = model[0].transform(X_test.iloc[[patient_nr]])[0]
contrib = pd.Series(model[-1].coef_[0] * x_scaled, index=X.columns)
top = contrib.reindex(contrib.abs().sort_values(ascending=False).index)[:8][::-1]

print(f"Predicted risk: {proba[patient_nr]:.0%}   |   actual label: {y_test.values[patient_nr]}")
top.plot.barh(color=np.where(top > 0, "tab:red", "tab:blue"))
plt.title("Top reasons for this patient"); plt.show()

### 🟡 Explore: is the model fair?
Does the model work equally well for women and men? Run the cell and compare.

In [ ]:
pred = (proba >= 0.5).astype(int)
test = df.loc[X_test.index].assign(pred=pred)

for sex, g in test.groupby("Sex"):
    ill, healthy = g[g.HeartDisease == 1], g[g.HeartDisease == 0]
    print(f"{sex}: n={len(g):3d} | ill={len(ill):3d} | "
          f"sensitivity={(ill.pred == 1).mean():.0%} | specificity={(healthy.pred == 0).mean():.0%}")

🟢 **Questions**
- How many **ill women** are in the test set? How much would you trust a sensitivity computed from so few patients?
- Who might be harmed if this model were used without checking subgroups?

<details><summary>Background</summary>

Small subgroups produce **unstable numbers** — one more or one fewer correct prediction changes the sensitivity a lot. Women are under-represented in many historical cardiology datasets, and symptoms can present differently. A model can look great *on average* and still fail a subgroup. That's why regulators and journals increasingly ask for **subgroup analyses**.
</details>

✍️ **Your answer:** *(double-click to write)*

…

### 🔴 Challenge: fix the cholesterol problem
Remember the impossible `Cholesterol == 0` values from Exercise 1? Replace them with the **median** of the real values, retrain, and compare accuracy and the cholesterol weight.

In [ ]:
# your code here

<details><summary>Solution</summary>

```python
df_clean = df.copy()
median_chol = df_clean.loc[df_clean.Cholesterol > 0, "Cholesterol"].median()
df_clean["Cholesterol"] = df_clean["Cholesterol"].replace(0, median_chol)

X2 = pd.get_dummies(df_clean.drop(columns="HeartDisease"), drop_first=True).astype(float)
X2_train, X2_test = X2.loc[X_train.index], X2.loc[X_test.index]
model2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X2_train, y_train)

print(f"Accuracy before: {model.score(X_test, y_test):.1%}  after: {model2.score(X2_test, y_test):.1%}")
print("Cholesterol weight before:", round(model[-1].coef_[0][list(X.columns).index('Cholesterol')], 2))
print("Cholesterol weight after: ", round(model2[-1].coef_[0][list(X2.columns).index('Cholesterol')], 2))
```
Before cleaning, "cholesterol = 0" was a hidden signal for *how/where the data was recorded* — not biology. That's a classic **shortcut**.
</details>

---
# ⭐ Bonus A · Mini-RAG: how an LLM gets "looked-up" knowledge
In the lecture we saw **Retrieval-Augmented Generation (RAG)**: before an LLM answers, the system **searches** a trusted document collection and puts the best passages into the prompt.
The search part is not magic — it's **similarity maths**. We build it here with 5 short medical texts. No API key needed.

This is a small **retrieval demo**, not a medical assistant: the notebook will show a sourced extract for the example question, then build a prompt for an optional LLM. It does not silently call a chatbot. Read the source before trusting either answer.

In [ ]:
import re, urllib.request
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE = "https://raw.githubusercontent.com/MoLue/wft_digital_medicine/main/medical_kb/"
chunks, sources = [], []
for i in range(1, 6):
    text = urllib.request.urlopen(f"{BASE}condition_{i}.txt").read().decode("utf-8")
    sentences = re.split(r"(?<=[.!?])\s+", " ".join(text.split()))
    for j in range(0, len(sentences), 3):                 # chunks of 3 sentences
        chunks.append(" ".join(sentences[j:j + 3]))
        sources.append(f"condition_{i}.txt")

vectorizer = TfidfVectorizer(stop_words="english")
chunk_vectors = vectorizer.fit_transform(chunks)
print(f"Knowledge base: {len(chunks)} chunks from {len(set(sources))} documents")

In [ ]:
def retrieve(question, k=5):
    scores = cosine_similarity(vectorizer.transform([question]), chunk_vectors)[0]
    best = scores.argsort()[::-1][:k]
    for rank, idx in enumerate(best, 1):
        print(f"#{rank}  score={scores[idx]:.2f}  [{sources[idx]}]\n    {chunks[idx][:250]}…\n")
    return [chunks[i] for i in best]

question = "What are the risk factors for coronary artery disease?"   # 🟡 change me
context = retrieve(question)

In [ ]:
# Show an answer for the worked example, using only a retrieved passage.
if question == "What are the risk factors for coronary artery disease?":
    evidence = next((c for c in context if "Risk factors include high blood pressure" in c), None)
    if evidence:
        answer = evidence.split("Risk factors include ", 1)[1].split(".", 1)[0]
        print(f"Evidence-backed extract (not an LLM answer): {answer}. [condition_4.txt]")
    else:
        print("No supporting passage retrieved; cannot answer from this context.")
else:
    print("No automatic answer for this question. Check the retrieved passages or ask a chatbot using the prompt below.")

prompt = f'''You are a careful medical assistant.
Answer the question ONLY using the context below. If the answer is not in the context, say "I don't know".
Cite the passage you used.

CONTEXT:
{chr(10).join('- ' + c for c in context)}

QUESTION: {question}'''
print("\nOptional chatbot prompt:\n", prompt)

🟢 **Check:** Is passage #1 really about *risk factors*? The worked example finds its answer in a lower-ranked passage. Retrieval can be wrong — and an LLM might then answer from the wrong context. The extract above is copied from a source, **not generated by an LLM**.

🟡 **Explore**
1. Copy the printed prompt into any chatbot you use and compare the answer with asking the question *without* context. (Never paste real patient data into public chatbots!)
2. Ask a question the knowledge base **cannot** answer, e.g. *"What is the dose of apixaban?"* Look at the scores. What should a safe system do now?

🔴 **Challenge:** TF-IDF only matches *words*. Try *"heart attack causes"* vs. *"myocardial infarction causes"*. Why might a real system use **embeddings** (meaning-based vectors) instead?

---
# ⭐ Bonus B · AI as your coding assistant ("English is the new programming language")
Colab has a built-in AI assistant (Gemini — look for the ✨ icon; availability depends on your account and region).
1. Create a new code cell and ask it: *"Plot MaxHR against Age from df, coloured by HeartDisease."*
2. **Before running:** read the generated code. Which columns does it use? Could it be wrong?
3. Run it. Then ask it something harder, e.g. *"Train a random forest and show the 5 most important features."*

🟢 **Reflect:** You just got working code without writing it. What do you still need to know to **trust** it? (Same question as for a clinical AI tool!)

---
# 🏁 Wrap-up (think about it, we'll discuss together)
1. What surprised you most today?
2. Name **one** question you will ask the next time someone presents you a "highly accurate" medical AI tool.
3. Where in your future clinical work could a model like this help — and where would it be dangerous?

### Want more?
All notebooks in this repository run in Colab without API keys:
- `medical_data_science_fundamentals.ipynb` — Python & pandas from scratch (beginner)
- `heart_disease_prediction_analysis.ipynb` — more models on this dataset (intermediate)
- `dm_nlp.ipynb` — NLP & transformers
- `llms_in_healthcare.ipynb` — LLMs, prompting & RAG (advanced)
- `agentic_ai_medical.ipynb` — AI agents in medicine (compact)

*Dataset: Heart Failure Prediction Dataset (fedesoriano, Kaggle), ODbL — combined from the UCI Cleveland, Hungarian, Switzerland, Long Beach VA and Statlog heart datasets.*